## 📦 1. Kutubxonalarni Yuklash

In [65]:
# Asosiy kutubxonalar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# ML kutubxonalari
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

# Sozlamalar
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

print("✅ Barcha kutubxonalar yuklandi!")

✅ Barcha kutubxonalar yuklandi!


## 📥 2. Ma'lumotlarni Yuklash

MovieLens 1M dataseti: 6,040 foydalanuvchi, 3,706 film, 1M reyting

In [66]:
# MovieLens datasetini yuklash
print("📥 Ma'lumotlar yuklanmoqda...\n")

# Filmlar
movies = pd.read_csv(
    '../datasets/ml-1m/movies.dat',
    sep='::',
    engine='python',
    encoding='latin-1',
    names=['MovieID', 'Title', 'Genres']
)

# Reytinglar
ratings = pd.read_csv(
    '../datasets/ml-1m/ratings.dat',
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp']
)

# Foydalanuvchilar
users = pd.read_csv(
    '../datasets/ml-1m/users.dat',
    sep='::',
    engine='python',
    names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
)

print(f"✅ Filmlar: {len(movies):,}")
print(f"✅ Reytinglar: {len(ratings):,}")
print(f"✅ Foydalanuvchilar: {len(users):,}")
print(f"\n📊 O'rtacha reyting: {ratings['Rating'].mean():.2f}/5")
print(f"🎯 Reyting oralig'i: {ratings['Rating'].min():.0f}-{ratings['Rating'].max():.0f}")

📥 Ma'lumotlar yuklanmoqda...

✅ Filmlar: 3,883
✅ Reytinglar: 1,000,209
✅ Foydalanuvchilar: 6,040

📊 O'rtacha reyting: 3.58/5
🎯 Reyting oralig'i: 1-5
✅ Filmlar: 3,883
✅ Reytinglar: 1,000,209
✅ Foydalanuvchilar: 6,040

📊 O'rtacha reyting: 3.58/5
🎯 Reyting oralig'i: 1-5


In [67]:
# Ma'lumotlarni ko'rish
print("📋 Birinchi filmlar:\n")
movies.head()


📋 Birinchi filmlar:



,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [68]:
print("\n📋 Birinchi reytinglar:\n")
ratings.head()


📋 Birinchi reytinglar:



,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 🔧 3. Ma'lumotlarni Tayyorlash

Film statistikasini hisoblash va datasetlarni birlashtirish

In [69]:
# Har bir film uchun statistika
movie_stats = ratings.groupby('MovieID').agg({
    'Rating': ['count', 'mean']
}).reset_index()
movie_stats.columns = ['MovieID', 'rating_count', 'rating_mean']

# Filmlar bilan birlashtirish
movies_full = movies.merge(movie_stats, on='MovieID', how='left')
movies_full['rating_count'] = movies_full['rating_count'].fillna(0)
movies_full['rating_mean'] = movies_full['rating_mean'].fillna(0)

print("✅ Ma'lumotlar birlashtirildi!")
print(f"\n📊 To'liq dataset: {len(movies_full)} ta film")
movies_full.head(10)

✅ Ma'lumotlar birlashtirildi!

📊 To'liq dataset: 3883 ta film


,MovieID,Title,Genres,rating_count,rating_mean
0,1,Toy Story (1995),Animation|Children's|Comedy,2077.0,4.146846
1,2,Jumanji (1995),Adventure|Children's|Fantasy,701.0,3.201141
2,3,Grumpier Old Men (1995),Comedy|Romance,478.0,3.016736
3,4,Waiting to Exhale (1995),Comedy|Drama,170.0,2.729412
4,5,Father of the Bride Part II (1995),Comedy,296.0,3.006757
5,6,Heat (1995),Action|Crime|Thriller,940.0,3.878723
6,7,Sabrina (1995),Comedy|Romance,458.0,3.410480
7,8,Tom and Huck (1995),Adventure|Children's,68.0,3.014706
8,9,Sudden Death (1995),Action,102.0,2.656863
9,10,GoldenEye (1995),Action|Adventure|Thriller,888.0,3.540541


In [71]:
# Kam reytingli filmlarni filtrlash (minimum 50 reyting)
MIN_RATINGS = 50
popular_movies = movies_full[movies_full['rating_count'] >= MIN_RATINGS].copy()

print(f"📊 {MIN_RATINGS}+ reyting olgan filmlar: {len(popular_movies)} ta")
print(f"🗑️  O'chirilgan filmlar: {len(movies_full) - len(popular_movies)} ta\n")

# Top 10 eng mashgur filmlar
print("🏆 TOP 10 ENG MASHGUR FILMLAR:\n")
top_movies = popular_movies.nlargest(10, 'rating_count')[['Title', 'rating_count', 'rating_mean']]
print(top_movies.to_string(index=False))

📊 50+ reyting olgan filmlar: 2514 ta
🗑️  O'chirilgan filmlar: 1369 ta

🏆 TOP 10 ENG MASHGUR FILMLAR:

                                                Title  rating_count  rating_mean
                               American Beauty (1999)        3428.0     4.317386
            Star Wars: Episode IV - A New Hope (1977)        2991.0     4.453694
Star Wars: Episode V - The Empire Strikes Back (1980)        2990.0     4.292977
    Star Wars: Episode VI - Return of the Jedi (1983)        2883.0     4.022893
                                 Jurassic Park (1993)        2672.0     3.763847
                           Saving Private Ryan (1998)        2653.0     4.337354
                    Terminator 2: Judgment Day (1991)        2649.0     4.058513
                                   Matrix, The (1999)        2590.0     4.315830
                            Back to the Future (1985)        2583.0     3.990321
                     Silence of the Lambs, The (1991)        2578.0     4.351823


## 🎯 4. Content-Based Filtering

Janr va tavsifga asoslangan tavsiyalar (o'xshash filmlarni topish)

In [72]:
# Janrlarni TF-IDF vektoriga aylantirish
print("🔨 Content-Based model qurilmoqda...\n")

# MUHIM: Index'ni reset qilish (0 dan boshlanishi uchun)
popular_movies = popular_movies.reset_index(drop=True)

# Janrlarni probel bilan ajratish (TF-IDF uchun)
popular_movies['genres_str'] = popular_movies['Genres'].str.replace('|', ' ')

# TF-IDF vektorlash
tfidf = TfidfVectorizer(stop_words='english') # and or not  
tfidf_matrix = tfidf.fit_transform(popular_movies['genres_str'])

# Kosinus o'xshashlik matritsasi
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"✅ TF-IDF matritsa o'lchami: {tfidf_matrix.shape}")
print(f"✅ O'xshashlik matritsasi: {cosine_sim.shape}")
print("\n💡 Content-Based model tayyor!")

🔨 Content-Based model qurilmoqda...

✅ TF-IDF matritsa o'lchami: (2514, 20)
✅ O'xshashlik matritsasi: (2514, 2514)

💡 Content-Based model tayyor!


In [73]:
# Content-Based tavsiya funksiyasi
def get_content_recommendations(title, top_n=10):
    """
    Film nomiga asoslangan o'xshash filmlarni qaytaradi
    
    Parametrlar:
    - title: Film nomi
    - top_n: Nechta tavsiya qaytarish kerak
    """
    # Film indeksini topish
    matches = popular_movies[popular_movies['Title'].str.contains(title, case=False, na=False)]
    
    if len(matches) == 0:
        # Agar topilmasa, eng yaqin nomli filmni topish
        all_titles = popular_movies['Title'].tolist()
        close_matches = [t for t in all_titles if title.lower() in t.lower() or t.lower() in title.lower()]
        
        if len(close_matches) > 0:
            # Eng birinchi mos keladiganini olish
            idx = popular_movies[popular_movies['Title'] == close_matches[0]].index[0]
            print(f"⚠️  '{title}' o'rniga '{close_matches[0]}' ishlatildi\n")
        else:
            # Umumiy eng mashgur filmlardan tavsiya
            print(f"⚠️  '{title}' topilmadi. Eng mashgur filmlar tavsiya etilmoqda:\n")
            return popular_movies.nlargest(top_n, 'rating_count')[['Title', 'Genres', 'rating_mean']]
    else:
        idx = matches.index[0]
    
    # O'xshashlik balllarini olish
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Ballar bo'yicha saralash
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Top N (o'zini hisobga olmasdan)
    sim_scores = sim_scores[1:top_n+1]
    
    # Film indekslarini olish
    movie_indices = [i[0] for i in sim_scores]
    
    # Natijalarni qaytarish
    recommendations = popular_movies.iloc[movie_indices][['Title', 'Genres', 'rating_mean']].copy()
    recommendations['similarity'] = [score[1] for score in sim_scores]
    
    return recommendations

print("✅ Content-Based tavsiya funksiyasi tayyor!")

✅ Content-Based tavsiya funksiyasi tayyor!


In [74]:
# Test qilish
print("🎬 TEST: 'Toy Story' filmiga o'xshash filmlar:\n")
recommendations = get_content_recommendations('Toy Story', top_n=10)
print(recommendations.to_string(index=False))

🎬 TEST: 'Toy Story' filmiga o'xshash filmlar:

                                         Title                      Genres  rating_mean  similarity
        Aladdin and the King of Thieves (1996) Animation|Children's|Comedy     2.894737    1.000000
                      American Tail, An (1986) Animation|Children's|Comedy     3.428218    1.000000
    American Tail: Fievel Goes West, An (1991) Animation|Children's|Comedy     2.830846    1.000000
                     Rugrats Movie, The (1998) Animation|Children's|Comedy     2.780142    1.000000
                          Bug's Life, A (1998) Animation|Children's|Comedy     3.854375    1.000000
                            Toy Story 2 (1999) Animation|Children's|Comedy     4.218927    1.000000
                            Chicken Run (2000) Animation|Children's|Comedy     3.879609    1.000000
Adventures of Rocky and Bullwinkle, The (2000) Animation|Children's|Comedy     2.495146    1.000000
                                  Balto (1995)       

In [75]:
# Yana bir test
print("🎬 TEST: 'Matrix' filmiga o'xshash filmlar:\n")
recommendations = get_content_recommendations('Matrix', top_n=10)
print(recommendations.to_string(index=False))

🎬 TEST: 'Matrix' filmiga o'xshash filmlar:

                            Title                 Genres  rating_mean  similarity
Terminator 2: Judgment Day (1991) Action|Sci-Fi|Thriller     4.058513         1.0
                      Solo (1996) Action|Sci-Fi|Thriller     2.113208         1.0
              Arrival, The (1996) Action|Sci-Fi|Thriller     3.151079         1.0
        Lawnmower Man, The (1992) Action|Sci-Fi|Thriller     2.660163         1.0
           Terminator, The (1984) Action|Sci-Fi|Thriller     4.152050         1.0
                  Face/Off (1997) Action|Sci-Fi|Thriller     3.401126         1.0
             Lost in Space (1998) Action|Sci-Fi|Thriller     2.584708         1.0
               Matrix, The (1999) Action|Sci-Fi|Thriller     4.315830         1.0
                  eXistenZ (1999) Action|Sci-Fi|Thriller     3.256098         1.0
             Deep Blue Sea (1999) Action|Sci-Fi|Thriller     2.872910         1.0


## 👥 5. Collaborative Filtering

Foydalanuvchilarning xatti-harakatiga asoslangan tavsiyalar (SVD algoritmi)

In [76]:
print("🔨 Collaborative Filtering model qurilmoqda...\n")

# Faqat mashgur filmlar uchun reytinglarni filtrlash
popular_movie_ids = popular_movies['MovieID'].values
ratings_filtered = ratings[ratings['MovieID'].isin(popular_movie_ids)].copy()

print(f"✅ Filtrlangan reytinglar: {len(ratings_filtered):,}")

# User-Item matritsani yaratish
user_movie_matrix = ratings_filtered.pivot_table(
    index='UserID',
    columns='MovieID',
    values='Rating'
).fillna(0)

print(f"✅ User-Item matritsa: {user_movie_matrix.shape}")
print(f"   ({user_movie_matrix.shape[0]} foydalanuvchi × {user_movie_matrix.shape[1]} film)")

# Sparsity hisoblash
sparsity = 1 - (len(ratings_filtered) / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1]))
print(f"\n🕳️  Matritsa siyrakligi: {sparsity*100:.1f}%")

🔨 Collaborative Filtering model qurilmoqda...

✅ Filtrlangan reytinglar: 977,839
✅ User-Item matritsa: (6040, 2514)
   (6040 foydalanuvchi × 2514 film)

🕳️  Matritsa siyrakligi: 93.6%
✅ User-Item matritsa: (6040, 2514)
   (6040 foydalanuvchi × 2514 film)

🕳️  Matritsa siyrakligi: 93.6%


In [77]:
# SVD (Singular Value Decomposition) qo'llash
print("⚙️  SVD algoritmi ishga tushirilmoqda...\n")

# Har bir foydalanuvchi uchun o'rtachani ayirish
user_ratings_mean = user_movie_matrix.mean(axis=1)
matrix_normalized = user_movie_matrix.sub(user_ratings_mean, axis=0)

# Sparse matritsa formatiga o'tkazish (tezlik uchun)
sparse_matrix = csr_matrix(matrix_normalized.values)

# SVD
k = 50  # latent faktorlar soni
U, sigma, Vt = svds(sparse_matrix, k=k)

# Sigma diagonalni matritsa qilish
sigma = np.diag(sigma)

# Bashorat qilingan reytinglar
predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.values.reshape(-1, 1)
predictions_df = pd.DataFrame(
    predicted_ratings,
    columns=user_movie_matrix.columns,
    index=user_movie_matrix.index
)

print(f"✅ SVD tugadi! (k={k} latent faktor)")
print(f"✅ Bashorat matritsasi: {predictions_df.shape}")
print("\n💡 Collaborative Filtering model tayyor!")

⚙️  SVD algoritmi ishga tushirilmoqda...

✅ SVD tugadi! (k=50 latent faktor)
✅ Bashorat matritsasi: (6040, 2514)

💡 Collaborative Filtering model tayyor!
✅ SVD tugadi! (k=50 latent faktor)
✅ Bashorat matritsasi: (6040, 2514)

💡 Collaborative Filtering model tayyor!


In [78]:
# Collaborative tavsiya funksiyasi
def get_collaborative_recommendations(user_id, top_n=10):
    """
    Foydalanuvchi uchun film tavsiya qiladi
    
    Parametrlar:
    - user_id: Foydalanuvchi ID
    - top_n: Nechta tavsiya qaytarish kerak
    """
    if user_id not in predictions_df.index:
        return f"❌ Foydalanuvchi {user_id} topilmadi!"
    
    # Foydalanuvchining bashorat reytinglari
    user_predictions = predictions_df.loc[user_id].sort_values(ascending=False)
    
    # Foydalanuvchi allaqachon ko'rgan filmlar
    user_watched = ratings_filtered[ratings_filtered['UserID'] == user_id]['MovieID'].values
    
    # Ko'rilmagan filmlarni filtrlash
    recommendations = user_predictions[~user_predictions.index.isin(user_watched)]
    
    # Top N
    top_recommendations = recommendations.head(top_n)
    
    # Film ma'lumotlarini qo'shish
    result = popular_movies[popular_movies['MovieID'].isin(top_recommendations.index)].copy()
    result['predicted_rating'] = result['MovieID'].map(top_recommendations)
    result = result.sort_values('predicted_rating', ascending=False)
    
    return result[['Title', 'Genres', 'rating_mean', 'predicted_rating']]


In [80]:
# Test qilish
test_user = 49
print(f"👤 Foydalanuvchi {test_user} uchun tavsiyalar:\n")

# Foydalanuvchi ko'rgan filmlar
user_watched = ratings_filtered[ratings_filtered['UserID'] == test_user].merge(
    movies, on='MovieID'
)[['Title', 'Rating']].sort_values('Rating', ascending=False).head(5)

print("📺 Ko'rgan filmlari (Top 5):\n")
print(user_watched.to_string(index=False))

print("\n\n🎯 Tavsiya etiladigan filmlar:\n")
recommendations = get_collaborative_recommendations(test_user, top_n=10)
print(recommendations.to_string(index=False))

👤 Foydalanuvchi 49 uchun tavsiyalar:

📺 Ko'rgan filmlari (Top 5):

                      Title  Rating
        Total Recall (1990)       5
Beauty and the Beast (1991)       5
            Die Hard (1988)       5
       Fugitive, The (1993)       5
    Schindler's List (1993)       5


🎯 Tavsiya etiladigan filmlar:

                                    Title                              Genres  rating_mean  predicted_rating
                        Braveheart (1995)                    Action|Drama|War     4.234957          3.556447
         Hunt for Red October, The (1990)                     Action|Thriller     4.052058          3.015801
                           Aladdin (1992) Animation|Children's|Comedy|Musical     3.788305          2.693719
           Godfather: Part II, The (1974)                  Action|Crime|Drama     4.357565          2.368203
                   Terminator, The (1984)              Action|Sci-Fi|Thriller     4.152050          2.349130
                          Face

## 🔀 6. Hybrid Recommender

Content-Based va Collaborative Filtering ni birlashtirish

In [60]:
def get_hybrid_recommendations(user_id, top_n=10, alpha=0.5):
    """
    Hybrid tavsiya tizimi
    
    Parametrlar:
    - user_id: Foydalanuvchi ID
    - top_n: Nechta tavsiya
    - alpha: Content-Based og'irligi (0-1, 0.5 = teng)
    
    Formula: hybrid_score = alpha × content + (1-alpha) × collaborative
    """
    if user_id not in predictions_df.index:
        return f"❌ Foydalanuvchi {user_id} topilmadi!"
    
    # Collaborative reytinglar
    collab_scores = predictions_df.loc[user_id]
    
    # Foydalanuvchi ko'rgan filmlar
    user_watched = ratings_filtered[ratings_filtered['UserID'] == user_id]['MovieID'].values
    
    # Foydalanuvchining sevimli janrlari
    user_genres = ratings_filtered[
        (ratings_filtered['UserID'] == user_id) & 
        (ratings_filtered['Rating'] >= 4)
    ].merge(movies, on='MovieID')['Genres'].str.split('|').explode().value_counts().head(3).index.tolist()
    
    # Content score: sevimli janrlar bilan mos kelish
    def content_score(genres):
        genre_list = genres.split('|')
        return sum(1 for g in user_genres if g in genre_list) / max(len(user_genres), 1)
    
    popular_movies['content_score'] = popular_movies['Genres'].apply(content_score)
    
    # Normalizatsiya
    collab_norm = (collab_scores - collab_scores.min()) / (collab_scores.max() - collab_scores.min())
    
    # Hybrid score hisoblash
    hybrid_scores = {}
    for movie_id in popular_movies['MovieID']:
        if movie_id not in user_watched and movie_id in collab_norm.index:
            content = popular_movies[popular_movies['MovieID'] == movie_id]['content_score'].values[0]
            collab = collab_norm[movie_id]
            hybrid_scores[movie_id] = alpha * content + (1 - alpha) * collab
    
    # Top N
    top_movies = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [m[0] for m in top_movies]
    
    # Natijalar
    result = popular_movies[popular_movies['MovieID'].isin(top_movie_ids)].copy()
    result['hybrid_score'] = result['MovieID'].map(dict(top_movies))
    result = result.sort_values('hybrid_score', ascending=False)
    
    return result[['Title', 'Genres', 'rating_mean', 'hybrid_score']]

print("✅ Hybrid tavsiya funksiyasi tayyor!")

✅ Hybrid tavsiya funksiyasi tayyor!


In [81]:
# Hybrid test
test_user = 40
print(f"🎯 HYBRID TAVSIYALAR - Foydalanuvchi {test_user}\n")
print("="*80)

# Foydalanuvchi sevimli janrlari
user_fav_genres = ratings_filtered[
    (ratings_filtered['UserID'] == test_user) & 
    (ratings_filtered['Rating'] >= 4)
].merge(movies, on='MovieID')['Genres'].str.split('|').explode().value_counts().head(3)

print("❤️  Sevimli janrlar:\n")
print(user_fav_genres)

print("\n\n🎬 Tavsiya etiladigan filmlar:\n")
hybrid_recs = get_hybrid_recommendations(test_user, top_n=10, alpha=0.5)
print(hybrid_recs.to_string(index=False))

🎯 HYBRID TAVSIYALAR - Foydalanuvchi 40

❤️  Sevimli janrlar:

Genres
Adventure    29
War          27
Action       23
Name: count, dtype: int64


🎬 Tavsiya etiladigan filmlar:

                                           Title                               Genres  rating_mean  hybrid_score
                        Where Eagles Dare (1969)                 Action|Adventure|War     3.811321      0.763161
       Star Wars: Episode IV - A New Hope (1977)      Action|Adventure|Fantasy|Sci-Fi     4.453694      0.673049
                            Jurassic Park (1993)              Action|Adventure|Sci-Fi     3.763847      0.664701
                                  Soldier (1998) Action|Adventure|Sci-Fi|Thriller|War     2.931034      0.664273
                     Boat, The (Das Boot) (1981)                     Action|Drama|War     4.302697      0.640192
                        Full Metal Jacket (1987)                     Action|Drama|War     4.110845      0.629241
Star Wars: Episode I - The Phanto

## 📊 7. Modellarni Taqqoslash

Uchala yondashuvni solishtirish

In [62]:
# Test foydalanuvchi
test_user = 50

print("🔍 3 TA MODEL TAQQOSLASH")
print("="*80)
print(f"Test foydalanuvchi: {test_user}\n")

# 1. Content-Based (foydalanuvchining eng yuqori baholagan filmiga o'xshash)
user_top_movies = ratings_filtered[
    ratings_filtered['UserID'] == test_user
].merge(movies, on='MovieID').sort_values('Rating', ascending=False)

# Mashgur filmlar ichidan birinchisini olish
for _, row in user_top_movies.iterrows():
    if row['MovieID'] in popular_movies['MovieID'].values:
        last_movie = row['Title']
        break
else:
    # Agar hech biri topilmasa, eng mashgurini olish
    last_movie = popular_movies.iloc[0]['Title']

print(f"🎬 Sevimli filmi: {last_movie}\n")
print("\n1️⃣ CONTENT-BASED (o'xshash filmlar):\n")
content_recs = get_content_recommendations(last_movie, top_n=5)
print(content_recs[['Title', 'Genres', 'similarity']].to_string(index=False))

# 2. Collaborative
print("\n\n2️⃣ COLLABORATIVE FILTERING (xatti-harakat):\n")
collab_recs = get_collaborative_recommendations(test_user, top_n=5)
print(collab_recs[['Title', 'Genres', 'predicted_rating']].to_string(index=False))

# 3. Hybrid
print("\n\n3️⃣ HYBRID (ikkalasi birlashgan):\n")
hybrid_recs = get_hybrid_recommendations(test_user, top_n=5, alpha=0.5)
print(hybrid_recs[['Title', 'Genres', 'hybrid_score']].to_string(index=False))

print("\n" + "="*80)

🔍 3 TA MODEL TAQQOSLASH
Test foydalanuvchi: 50

🎬 Sevimli filmi: American Beauty (1999)


1️⃣ CONTENT-BASED (o'xshash filmlar):

⚠️  'American Beauty (1999)' o'rniga 'American Beauty (1999)' ishlatildi

                       Title       Genres  similarity
           To Die For (1995) Comedy|Drama         1.0
Kicking and Screaming (1995) Comedy|Drama         1.0
 Doom Generation, The (1995) Comedy|Drama         1.0
      Unstrung Heroes (1995) Comedy|Drama         1.0
     Boys on the Side (1995) Comedy|Drama         1.0


2️⃣ COLLABORATIVE FILTERING (xatti-harakat):

               Title                 Genres  predicted_rating
 Patriot, The (2000)       Action|Drama|War          1.905030
High Fidelity (2000)                 Comedy          1.822410
        U-571 (2000)        Action|Thriller          1.588670
  Stand by Me (1986) Adventure|Comedy|Drama          1.436730
Shanghai Noon (2000)                 Action          1.345396


3️⃣ HYBRID (ikkalasi birlashgan):

                

## 💾 8. Modellarni Saqlash

Deployment uchun modellarni saqlash

In [63]:
import pickle
import os

# Models papkasini yaratish
os.makedirs('../models', exist_ok=True)

# Modellarni saqlash
print("💾 Modellar saqlanmoqda...\n")

# 1. Content-Based
with open('../models/content_model.pkl', 'wb') as f:
    pickle.dump({
        'movies': popular_movies,
        'cosine_sim': cosine_sim,
        'tfidf_matrix': tfidf_matrix
    }, f)
print("✅ content_model.pkl")

# 2. Collaborative
with open('../models/collaborative_model.pkl', 'wb') as f:
    pickle.dump({
        'predictions': predictions_df,
        'user_movie_matrix': user_movie_matrix,
        'movies': popular_movies
    }, f)
print("✅ collaborative_model.pkl")

# 3. Ma'lumotlar
with open('../models/data.pkl', 'wb') as f:
    pickle.dump({
        'movies': movies,
        'ratings': ratings_filtered,
        'popular_movies': popular_movies
    }, f)
print("✅ data.pkl")

print("\n🎉 Barcha modellar saqlandi! (../models/)")

💾 Modellar saqlanmoqda...

✅ content_model.pkl
✅ collaborative_model.pkl
✅ data.pkl

🎉 Barcha modellar saqlandi! (../models/)
✅ collaborative_model.pkl
✅ data.pkl

🎉 Barcha modellar saqlandi! (../models/)


## 🎉 9. Yakuniy Natijalar

### ✅ Yaratilgan:
1. **Content-Based Model** - Janrga asoslangan tavsiyalar
2. **Collaborative Filtering** - Foydalanuvchi harakatiga asoslangan (SVD)
3. **Hybrid Model** - Ikkalasini birlashtirgan (eng yaxshi natija)

### 📊 Natijalar:
- ✅ 1,800+ mashgur filmlar (50+ reyting)
- ✅ 6,000+ foydalanuvchi
- ✅ 950,000+ reyting
- ✅ 3 xil tavsiya algoritmi

### 🚀 Keyingi Qadamlar:
1. Streamlit web ilovasi yaratish
2. Flask/FastAPI REST API
3. Docker konteynerizatsiya
4. Cloud deployment (Heroku/AWS)

---

**🎬 Tabriklaymiz! Netflix tavsiya tizimi tayyor!** 🎉

In [64]:
# Final demo
print("🎬 FINAL DEMO - Interaktiv Tavsiya Tizimi")
print("="*80)

# Tasodifiy foydalanuvchi
random_user = np.random.choice(predictions_df.index)

print(f"\n👤 Tasodifiy foydalanuvchi: {random_user}")
print("\n📺 Ko'rgan filmlari (oxirgi 5 ta):")
watched = ratings_filtered[
    ratings_filtered['UserID'] == random_user
].merge(movies, on='MovieID').sort_values('Timestamp', ascending=False).head(5)
print(watched[['Title', 'Rating', 'Genres']].to_string(index=False))

print("\n\n🎯 SIZ UCHUN TAVSIYALAR (Hybrid Model):\n")
final_recs = get_hybrid_recommendations(random_user, top_n=10, alpha=0.5)
print(final_recs[['Title', 'Genres', 'rating_mean']].to_string(index=False))

print("\n" + "="*80)
print("✅ LOYIHA MUVAFFAQIYATLI YAKUNLANDI!")
print("="*80)

🎬 FINAL DEMO - Interaktiv Tavsiya Tizimi

👤 Tasodifiy foydalanuvchi: 3613

📺 Ko'rgan filmlari (oxirgi 5 ta):
                         Title  Rating                   Genres
                Serpico (1973)       5              Crime|Drama
Godfather: Part II, The (1974)       5       Action|Crime|Drama
             Sting, The (1973)       5             Comedy|Crime
             True Crime (1999)       4           Crime|Thriller
       Mulholland Falls (1996)       4 Crime|Film-Noir|Thriller


🎯 SIZ UCHUN TAVSIYALAR (Hybrid Model):

                                                Title                        Genres  rating_mean
                             Big Lebowski, The (1998) Comedy|Crime|Mystery|Thriller     3.738377
       Midnight in the Garden of Good and Evil (1997)    Comedy|Crime|Drama|Mystery     3.253185
                                  Jackie Brown (1997)                   Crime|Drama     3.689227
Man Bites Dog (C'est arrivé près de chez vous) (1992)     Action|Comedy|Crime